# Extraer información con Gemini (compatible con Colab)
Este notebook funciona en Google Colab y también en local.

En Colab:
1. Agrega tu clave en **Secrets** con el nombre `GEMINI_API_KEY`.
2. Sube el CSV a `/content` o ajusta la ruta en la celda de carga de datos.

In [ ]:
# Instala dependencias cuando corres en Colab
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip -q install -U google-genai pandas

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai


def get_api_key() -> str:
    # Prioriza Secrets de Colab y usa .env como fallback local.
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass

    load_dotenv()
    return os.getenv("GEMINI_API_KEY", "")


api_key = get_api_key()
if not api_key:
    raise ValueError(
        "No se encontro GEMINI_API_KEY. En Colab configuralo en Secrets; en local usa .env."
    )

In [ ]:
# Ruta compatible con Colab (/content) y con entorno local.
if "IN_COLAB" in globals() and IN_COLAB:
    csv_path = Path("/content/Monitoreo noticias with diffbot fields.csv")
else:
    base_dir = Path.cwd()
    csv_path = base_dir / ".." / "data" / "Monitoreo noticias with diffbot fields.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"No existe el CSV en: {csv_path}")

df = pd.read_csv(csv_path)
print(f"CSV: {csv_path}")
print(f"Number of rows in DataFrame: {len(df)}")

Number of rows in DataFrame: 700


In [ ]:
MODEL_NAME = "gemini-3.5-flash-lite"

n_i = 0
n_f = 5

question = "Extrae los nombres de las personas mencionadas en el artículo. Devuelve la respuesta en formato JSON con una lista de nombres bajo la clave 'nombres'. Si no se pueden determinar nombres, devuelve un JSON con una lista vacía. Incluye el ID del artículo en el objeto JSON."

client = genai.Client(api_key=api_key)

for index, row in df.iloc[n_i:n_f].iterrows():
    article_id = str(row["ID"]).strip()
    html_code = str(row["diffbot_html"]).strip()
    if not html_code or html_code == "nan":
        print(f"Row {index} (ID: {article_id}) has no HTML code. Skipping.")
        continue

    prompt = f"Responde en español. {question}\n\nID del artículo: {article_id}\nHTML: {html_code}"
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
    )

    response_text = response.text or ""
    # response_tokens = count_tokens(client, MODEL_NAME, response_text)

    print(f"\n[{index}] Article ID: {article_id}")
    print("-" * 80)
    print(response_text)
    print("-" * 80)
